<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/3_covariance_matrix_adaptation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 协方差矩阵自适应（Covariance Matrix Adaptation）

连续优化中的一个重要困难是病态尺度（ill-conditioning）：不同变量对目标函数的影响尺度差异很大。从二阶局部结构看，这对应于 Hessian 矩阵最大特征值与最小特征值之比很大，这个比值称为条件数（Condition Number）。

本章主要验证两点：
* 对病态尺度函数，仅进行步长自适应仍然效率较低。
* 通过 Covariance Matrix Adaptation 自适应搜索分布（正态分布）的协方差矩阵，也就是自适应分布的形状，可以显著提高优化效率。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#### CSA-ES

In [ ]:
class CSAES(object):
    """CSA Evolution Strategy"""

    def __init__(self, func, init_mean, init_sigma, nsample):
        """构造函数

        Parameters
        ----------
        func : callable
            目标函数（最小化）
        init_mean : ndarray (1D)
            初始均值向量
        init_sigma : float
            初始步长
        nsample : int
            样本数量
        """
        self.func = func
        self.mean = init_mean
        self.sigma = init_sigma
        self.N = self.mean.shape[0]                     # 搜索空间维数
        self.arx = np.zeros((nsample, self.N)) * np.nan # 候选解
        self.arf = np.zeros(nsample) * np.nan           # 候选解的目标函数值

        self.weights = np.zeros(nsample)
        self.weights[:nsample//4] = 1.0 / (nsample//4)  # 权重，总和为 1

        # For CSA
        self.ps = np.zeros(self.N)
        self.cs = 4.0 / (self.N + 4.0)
        self.ds = 1.0 + self.cs
        self.chiN = np.sqrt(self.N) * (1.0 - 1.0 / (4.0 * self.N) + 1.0 / (21.0 * self.N * self.N))
        self.mueff = 1.0 / np.sum(self.weights**2)

    def sample(self):
        """生成候选解。"""
        self.arx = self.mean + self.sigma * np.random.normal(size=self.arx.shape)

    def evaluate(self):
        """评估候选解。"""
        for i in range(self.arf.shape[0]):
            self.arf[i] = self.func(self.arx[i])

    def update_mean(self):
        """更新均值向量。"""
        idx = np.argsort(self.arf)  # idx[i] 是目标函数值排名第 i 的候选解索引
        self.mean += np.dot(self.weights, (self.arx[idx] - self.mean))

    def update_sigma(self):
        """CSA"""
        idx = np.argsort(self.arf)  # idx[i] 是目标函数值排名第 i 的候选解索引
        # 更新进化路径（累积均值向量移动）
        self.ps = (1 - self.cs) * self.ps + np.sqrt(self.cs * (2 - self.cs) * self.mueff) * np.dot(self.weights, (self.arx[idx] - self.mean)) / self.sigma
        # 若进化路径长度大于随机函数下的期望，则增大步长
        self.sigma = self.sigma * np.exp(self.cs / self.ds * (np.linalg.norm(self.ps) / self.chiN - 1))

#### Ellipsoid 函数与条件数

In [ ]:
EllipsoidCondition = 1
def ellipsoid(x):
    """各分量平方的加权和，最优解为 (0,...,0)。"""
    w = np.logspace(0, EllipsoidCondition, base=10, num=x.shape[0], endpoint=True)
    return np.sqrt(np.sum(w * x ** 2))

绘制等高线，并将条件数参数依次改为 0、1、2、……，观察等高线形状的变化。

In [ ]:
dx, dy = 0.05, 0.05
y, x = np.mgrid[slice(-1, 1 + dy, dy), slice(-1, 1 + dx, dx)]
z = np.zeros(x.shape)
for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        z[i,j] = ellipsoid(np.array([x[i,j], y[i,j]]))

plt.figure()
CS = plt.contour(x, y, z)
plt.clabel(CS, inline=1, fontsize=10)
plt.axis('equal')

#### 使用 CSA-ES 进行实验

先设置 `EllipsoidCondition = 6`，然后执行下面的实验。

In [ ]:
es = CSAES(func=ellipsoid,
           init_mean=np.ones(10),
           init_sigma=1,
           nsample=10)

maxiter = 2000
mean = np.zeros(maxiter) * np.nan
sigmaN = np.zeros(maxiter) * np.nan
for i in range(maxiter):
    es.sample()
    es.evaluate()
    es.update_sigma()
    es.update_mean()
    mean[i] = es.func(es.mean)
    sigmaN[i] = es.sigma * es.N

In [ ]:
plt.semilogy(mean, '-b', label='f(mean)')
plt.semilogy(sigmaN, '--g', label='sigma*N')
plt.xlabel('no. of iterations', fontsize='large')
plt.grid()
plt.legend()

#### 改变条件数时收敛曲线的变化

In [ ]:
EllCondArray = np.linspace(0, 20, num=21, endpoint=True)
for j in range(EllCondArray.shape[0]):
    EllipsoidCondition = EllCondArray[j]

    def ellipsoid(x):
        """各分量平方的加权和，最优解为 (0,...,0)。"""
        w = np.logspace(0, EllipsoidCondition, base=2, num=x.shape[0], endpoint=True)
        return np.sqrt(np.sum(w * x ** 2))

    es = CSAES(func=ellipsoid,
               init_mean=np.zeros(20),
               init_sigma=1,
               nsample=10)

    maxiter = 5000
    mean = np.zeros(maxiter) * np.nan
    sigmaN = np.zeros(maxiter) * np.nan
    for i in range(maxiter):
        es.sample()
        es.evaluate()
        es.update_sigma()
        es.update_mean()
        mean[i] = es.func(es.mean)
        sigmaN[i] = es.sigma * es.N
        if mean[i] < 1e-10:
            break

    plt.loglog(mean, label=r'Cond=$2^{'+str(j)+'}$')

plt.xlabel('no. of iterations', fontsize='large')
plt.grid()
plt.legend(loc='best', fontsize='small', ncol=2)


随着条件数增大，目标函数在对数尺度上的收敛曲线斜率变得更缓，优化所需时间明显增加。

* 请结合生成候选解的分布形状与目标函数等高线之间的关系进行分析。

#### 协方差矩阵自适应（Covariance Matrix Adaptation）

在介绍一般的协方差矩阵自适应之前，先看只学习协方差矩阵对角元素的 Separable-CMA [1]。

[1] Raymond Ros, Nikolaus Hansen. A Simple Modification in CMA-ES Achieving Linear Time and Space Complexity. 10th International Conference on Parallel Problem Solving From Nature, Sep 2008

#### 基本思想

让候选解生成分布的协方差矩阵自适应变化，使不同坐标方向具有不同搜索尺度，并让搜索分布的形状逐渐接近目标函数等高线的形状。

#### 具体更新方法

设协方差矩阵写成 $\Sigma=\sigma^2C$。这里 $C$ 是对角矩阵（代码中用 `D` 表示）。每一代按 $x_i\sim m+\sigma\sqrt{C}\mathcal{N}(0,I)\sim\mathcal{N}(m,\sigma^2C)$ 生成候选解。评估后按目标函数值升序排序，记第 $i$ 个优秀解为 $x_{i:\lambda}$。于是第 $k$ 个对角元素按
$$
[C]_{k,k} \leftarrow (1 - c_\mu)[C]_{k,k} + c_\mu \sum_{i=1}^{\lambda} w_i \left(\frac{[x_{i:\lambda} - m]_k}{\sigma}\right)^2
$$
更新。$[\cdot]_k$ 和 $[\cdot]_{k,k}$ 分别表示向量的第 $k$ 个元素和矩阵的 $(k,k)$ 元素。注意右侧的 $m$、$\sigma$、$C$ 都是生成当前候选解时的旧参数。

#### 解释

例如令 $w_1=\dots=w_\mu=1/\mu$、$w_{\mu+1}=\dots=w_\lambda=0$，其中 $\mu=\lfloor\lambda/4\rfloor$，那么这一更新可以看作：在 $m$、$\sigma$ 已知时，用排名前四分之一的候选解对 $C$ 做最大似然估计。另一个解释是，该更新对应于目标函数关于正态分布参数的自然梯度方向 [2]。

[2] Akimoto, Y., Nagata, Y., Ono, I. et al. Theoretical Foundation for CMA-ES from Information Geometry Perspective. Algorithmica 64, 698–716 (2012). https://doi.org/10.1007/s00453-011-9564-8

#### 验证无偏性

与 CSA 的设计思想相同：如果目标函数完全随机，就没有可利用的信息，因此分布参数在平均意义上不应被系统性改变。下面验证上述协方差更新具有这一无偏性。

若 $x_{i:\lambda}$ 独立服从 $\mathcal{N}(m,\sigma^2C)$，则
$$
\frac{x_{i:\lambda}-m}{\sigma}\sim\mathcal{N}(0,C).
$$
因此更新式第二项的期望为
$$\begin{aligned}
\mathbb{E}\left[\sum_{i=1}^{\lambda}w_i\left(\frac{[x_{i:\lambda}-m]_k}{\sigma}\right)^2\right]
&=\sum_{i=1}^{\lambda}w_i\mathbb{E}\left[\left(\frac{[x_{i:\lambda}-m]_k}{\sigma}\right)^2\right]\\
&=\sum_{i=1}^{\lambda}w_i[C]_{k,k}\\
&=[C]_{k,k}.
\end{aligned}$$
所以更新后 $[C]_{k,k}$ 的期望恰好等于更新前的值，满足随机目标函数下的无偏性要求。

In [ ]:
class CMAES(object):
    """带 CSA 的 CMA Evolution Strategy。"""

    def __init__(self, func, init_mean, init_sigma, nsample):
        """构造函数

        Parameters
        ----------
        func : callable
            目标函数（最小化）
        init_mean : ndarray (1D)
            初始均值向量
        init_sigma : float
            初始步长
        nsample : int
            样本数量
        """
        self.func = func
        self.mean = init_mean
        self.sigma = init_sigma
        self.N = self.mean.shape[0]                     # 搜索空间维数
        self.arx = np.zeros((nsample, self.N)) * np.nan # 候选解
        self.arf = np.zeros(nsample) * np.nan           # 候选解的目标函数值
        self.D = np.ones(self.N)

        self.weights = np.zeros(nsample)
        self.weights[:nsample//4] = 1.0 / (nsample//4)  # 权重，总和为 1

        # For CSA
        self.ps = np.zeros(self.N)
        self.cs = 4.0 / (self.N + 4.0)
        self.ds = 1.0 + self.cs
        self.chiN = np.sqrt(self.N) * (1.0 - 1.0 / (4.0 * self.N) + 1.0 / (21.0 * self.N * self.N))
        self.mueff = 1.0 / np.sum(self.weights**2)

        # For CMA
        self.cmu = self.mueff / (4 * self.N + self.mueff)

    def sample(self):
        """生成候选解。"""
        self.arx = self.mean + self.sigma * np.random.normal(size=self.arx.shape) * np.sqrt(self.D)

    def evaluate(self):
        """评估候选解。"""
        for i in range(self.arf.shape[0]):
            self.arf[i] = self.func(self.arx[i])

    def update_param(self):
        """更新参数。"""
        idx = np.argsort(self.arf)  # idx[i] 是目标函数值排名第 i 的候选解索引
        # 更新进化路径（累积均值向量移动）
        self.ps = (1 - self.cs) * self.ps + np.sqrt(self.cs * (2 - self.cs) * self.mueff) * np.dot(self.weights, (self.arx[idx] - self.mean)) / np.sqrt(self.D) / self.sigma

        # 更新协方差矩阵的对角元素
        self.D = (1 - self.cmu) * self.D + self.cmu * np.dot(self.weights, (self.arx[idx] - self.mean) ** 2) / self.sigma ** 2

        # 若进化路径长度大于随机函数下的期望，则增大步长
        self.sigma = self.sigma * np.exp(self.cs / self.ds * (np.linalg.norm(self.ps) / self.chiN - 1))
        self.mean += np.dot(self.weights, (self.arx[idx] - self.mean))

In [ ]:
EllipsoidCondition = 6
def ellipsoid(x):
    """各分量平方的加权和，最优解为 (0,...,0)。"""
    w = np.logspace(0, EllipsoidCondition, base=10, num=x.shape[0], endpoint=True)
    return np.sqrt(np.sum(w * x ** 2))

es = CMAES(func=ellipsoid,
           init_mean=np.zeros(10),
           init_sigma=1,
           nsample=10)

maxiter = 1000
mean = np.zeros(maxiter) * np.nan
sigmaN = np.zeros(maxiter) * np.nan
diagC = np.zeros((maxiter, es.N)) * np.nan

for i in range(maxiter):
    es.sample()
    es.evaluate()
    es.update_param()
    mean[i] = es.func(es.mean)
    sigmaN[i] = es.sigma * es.N
    diagC[i] = es.D

plt.subplot(211)
plt.semilogy(mean, '-b', label='f(mean)')
plt.semilogy(sigmaN, '--g', label='sigma*N')
plt.xlabel('no. of iterations', fontsize='large')
plt.grid()
plt.legend(loc='best')
plt.subplot(212)
plt.semilogy(diagC)
plt.xlabel('no. of iterations', fontsize='large')
plt.ylabel('D[i]')
plt.grid()


#### 比较不同条件数

In [ ]:
EllCondArray = np.linspace(0, 6, num=13, endpoint=True)
for j in range(EllCondArray.shape[0]):
    EllipsoidCondition = EllCondArray[j]

    def ellipsoid(x):
        """各分量平方的加权和，最优解为 (0,...,0)。"""
        w = np.logspace(0, EllipsoidCondition, base=10, num=x.shape[0], endpoint=True)
        return np.sqrt(np.sum(w * x ** 2))

    es = CMAES(func=ellipsoid,
               init_mean=np.zeros(10),
               init_sigma=1,
               nsample=10)

    maxiter = 1000
    mean = np.zeros(maxiter) * np.nan
    sigmaN = np.zeros(maxiter) * np.nan
    for i in range(maxiter):
        es.sample()
        es.evaluate()
        es.update_param()
        mean[i] = es.func(es.mean)
        sigmaN[i] = es.sigma * es.N

    plt.semilogy(mean, label='Cond=1e'+str(EllipsoidCondition))

plt.xlabel('no. of iterations', fontsize='large')
plt.grid()
plt.legend(loc='best', fontsize='small', ncol=2)

当条件数较大时：
* 搜索初期需要额外时间学习协方差矩阵。
* 一旦协方差矩阵学习完成，之后收敛曲线的斜率几乎不再依赖条件数。

## 思考

* 研究学习完成后的协方差矩阵与目标函数局部几何之间的关系。
* 思考协方差矩阵的形状是通过什么机制逐渐适应目标函数的。
* 为什么协方差矩阵没有被正确学习时（例如只有 CSA-ES），在病态尺度问题上收敛会很慢？可以分别可视化 CSA-ES 与 CMA-ES 中均值向量到最优解的距离和步长，分析两个问题：为什么相对于最优解距离，步长会被压得很小；以及步长变小后为什么导致优化速度下降。